In [5]:
from astropy.io import fits

# Open the FITS file
fits_file = "iso_6.00.fits"
with fits.open(fits_file) as hdul:
    # Print structure of the file
    print("FITS structure:")
    hdul.info()

    # Inspect column names in the first extension (usually index 1)
    data = hdul[1].data
    print("\nColumn names in the FITS file:")
    print(data.columns.names)

FITS structure:
Filename: iso_6.00.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     18   30R x 5C   [D, D, D, D, D]   

Column names in the FITS file:
['Mass', 'Teff', 'logL', 'logG', 'Rad']


In [6]:
import numpy as np
from astropy.io import fits
from astropy.table import Table, vstack
from scipy.interpolate import CubicSpline
import math

# === Load the FITS isochrone ===
input_fits = "iso_6.00.fits"
hdul = fits.open(input_fits)
data = hdul[1].data

# === Extract and convert columns ===
mass = data['Mass']
logT = np.log10(data['Teff'])
logL = data['logL']
logg = data['logG']

# === Interpolation range ===
interp_mask = (mass >= 0.1) & (mass <= 0.7)
mass_interp = mass[interp_mask]
logT_interp = logT[interp_mask]
logL_interp = logL[interp_mask]
logg_interp = logg[interp_mask]

# Fit cubic splines
spline_logT = CubicSpline(mass_interp, logT_interp)
spline_logL = CubicSpline(mass_interp, logL_interp)
spline_logg = CubicSpline(mass_interp, logg_interp)

# Generate new masses to insert
new_mass_vals = np.linspace(0.1, 0.7, 100)
new_mass_vals = np.setdiff1d(new_mass_vals, mass, assume_unique=True)

# Interpolate values
new_logT = spline_logT(new_mass_vals)
new_logL = spline_logL(new_mass_vals)
new_logg = spline_logg(new_mass_vals)

# Construct remaining columns
new_logT_WR = new_logT
new_M_act = new_mass_vals
new_phase = np.ones_like(new_mass_vals, dtype=int)
new_source = np.array(['Baraffe'] * len(new_mass_vals))

# Build new table with updated structure
new_table = Table([
    new_mass_vals,
    new_logT,
    new_logL,
    new_logg,
    new_logT_WR,
    new_M_act,
    new_phase,
    new_source
], names=('M_ini', 'logT', 'logL', 'logg', 'logT_WR', 'M_act', 'phase', 'source'))

# Save as .dat file
output_dat = "Baraffe_1Myr_interpolated.dat"
with open(output_dat, "w") as f:
    f.write("# M_init       log T       log L       log g    logT_WR       M_curr phase Source\n")
    f.write("# (Msun)    (Kelvin)      (Lsun)       (cgs)   (Kelvin)       (Msun)    () ()\n")
    for row in new_table:
        f.write(f"{row['M_ini']:.6f} {row['logT']:.4f} {row['logL']:.4f} {row['logg']:.4f} "
                f"{row['logT_WR']:.4f} {row['M_act']:.6f} {row['phase']} {row['source']}\n")

print(f"Interpolated .dat file written to: {output_dat}")

# === Save full combined FITS ===
# Rebuild original table with new column names to match new data
orig_logT = np.log10(data['Teff'])
orig_table = Table([
    data['Mass'],
    orig_logT,
    data['logL'],
    data['logG'],
    orig_logT,
    data['Mass'],
    np.ones_like(data['Mass'], dtype=int),
    np.array(['Baraffe'] * len(data['Mass']))
], names=('M_ini', 'logT', 'logL', 'logg', 'logT_WR', 'M_act', 'phase', 'source'))

combined_table = vstack([orig_table, new_table])
combined_table.sort('M_ini')

output_fits = "Baraffe_1Myr_interpolated.fits"
combined_table.write(output_fits, format='fits', overwrite=True)
print(f"Interpolated FITS file written to: {output_fits}")

Interpolated .dat file written to: Baraffe_1Myr_interpolated.dat
Interpolated FITS file written to: Baraffe_1Myr_interpolated.fits
